# Setup

In [1]:
import duckdb
con = duckdb.connect("../data/yelp.duckdb")

In [2]:
con.sql("SHOW TABLES").df()

,name
0,business_rag
1,business_raw
2,business_restaurants
3,checkin_rag
4,review_rag
5,review_stats
6,tip_rag
7,user_rag
8,user_stats


# Restaurants and categories

## Q1. most common categories

In [13]:
con.sql("""
SELECT TRIM(category) AS category,
       COUNT(*) AS num_restaurants,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM business_restaurants), 1) AS pct_of_restaurants
FROM (
    SELECT UNNEST(string_split(categories, ',')) AS category
    FROM business_restaurants
)
WHERE TRIM(category) NOT IN ('Restaurants', 'Food', 'Nightlife', 'Bars', 'Event Planning & Services')
GROUP BY category
ORDER BY num_restaurants DESC
LIMIT 20
""").df()

,category,restaurants,pct_of_restaurants
0,Sandwiches,8366,16.0
1,American (Traditional),8139,15.6
2,Pizza,7093,13.6
3,Fast Food,6472,12.4
4,Breakfast & Brunch,6239,11.9
5,American (New),6097,11.7
6,Burgers,5636,10.8
7,Mexican,4600,8.8
8,Italian,4573,8.7
9,Coffee & Tea,4053,7.8


Of the 52,268 restaurants, the most common cuisine and food-style tags are
Sandwiches (8,366, 16.0%), American (Traditional) (8,139, 15.6%), Pizza (7,093, 13.6%),
Fast Food (6,472, 12.4%) and Breakfast & Brunch (6,239, 11.9%). Among specific cuisines,
Mexican (4,600, 8.8%) and Italian (4,573, 8.7%) are the most common, followed by
Chinese (3,169, 6.1%).

**Notes**
- `Restaurants` is on all 52,268 rows (100%), as expected, because the table was filtered by
  that tag. It is a check that the filter worked, not a finding.
- I left out generic tags that don't describe a cuisine: Restaurants, Food, Nightlife, Bars
  and Event Planning & Services.
- A restaurant has several tags, so it is counted in each one, and the percentages add up to
  more than 100%.

In [31]:
import pandas as pd
pd.set_option("display.max_rows", 100)

## Q2. top 5 cuisines per city, as a share

In [32]:
con.sql("""
WITH categories AS (
    SELECT city, state, TRIM(category) AS category
    FROM (
        SELECT city, state, UNNEST(string_split(categories, ',')) AS category
        FROM business_restaurants
    )
    WHERE TRIM(category) NOT IN ('Restaurants', 'Food', 'Nightlife', 'Bars', 'Event Planning & Services')
),
restaurant_per_city AS (
    SELECT city, state, COUNT(*) AS city_restaurants
    FROM business_restaurants
    GROUP BY city, state
    HAVING COUNT(*) >= 500
)
SELECT c.city, c.state, c.category,
       COUNT(*) AS restaurants,
       ROUND(100.0 * COUNT(*) / MAX(t.city_restaurants), 1) AS pct_of_city
FROM categories c
JOIN restaurant_per_city t USING (city, state)
GROUP BY c.city, c.state, c.category
QUALIFY ROW_NUMBER() OVER (PARTITION BY c.city, c.state ORDER BY COUNT(*) DESC) <= 5
ORDER BY MAX(t.city_restaurants) DESC, c.city, restaurants DESC
""").df()

,city,state,category,restaurants,pct_of_city
0,Philadelphia,PA,Sandwiches,929,15.9
1,Philadelphia,PA,Pizza,800,13.7
2,Philadelphia,PA,American (New),751,12.8
3,Philadelphia,PA,Breakfast & Brunch,686,11.7
4,Philadelphia,PA,American (Traditional),663,11.3
5,Tampa,FL,Sandwiches,472,15.9
6,Tampa,FL,American (Traditional),425,14.4
7,Tampa,FL,American (New),348,11.8
8,Tampa,FL,Fast Food,342,11.6
9,Tampa,FL,Breakfast & Brunch,332,11.2


**Answer.** In the 15 cities with at least 500 restaurants, the top 5 tags are similar from
city to city. Sandwiches is at or near the top in most of the cities I checked (15.9% in
Philadelphia and Tampa, 15.2% in Indianapolis and Nashville, 19.6% in St. Louis), and
American (Traditional), American (New), Breakfast & Brunch and Fast Food come up often.
A few cities stand out. In Tucson, Mexican is 20.7% of restaurants, about 2.4 times its
overall share of 8.8%. In Metairie, Seafood is 15.7%, about 2.3 times its overall share of
6.8%. In Indianapolis, Mexican is 13.3%, about 1.5 times its overall share.

**Notes**
- Only cities with at least 500 restaurants are included, because shares from tiny cities
  are unstable.
- The share is the number of restaurants with the tag divided by that city's total
  restaurants. A restaurant has several tags, so shares in one city add up to more than 100%.
- I left out generic tags (Restaurants, Food, Nightlife, Bars, Event Planning & Services).
- `city` and `state` are used together, because the same city name can exist in two states.

## Q3. highest average stars by category

In [39]:
con.sql("""
WITH categories AS (
    SELECT TRIM(category) AS category, stars
    FROM (
        SELECT UNNEST(string_split(categories, ',')) AS category, stars
        FROM business_restaurants
    )
    WHERE TRIM(category) NOT IN ('Restaurants', 'Food', 'Nightlife', 'Bars', 'Event Planning & Services')
)
SELECT  category,
        COUNT(*) AS restaurants,
        ROUND(AVG(stars),2) as avg_stars
FROM categories 
GROUP BY category
HAVING COUNT(*) >= 100
ORDER BY avg_stars DESC, restaurants DESC
""").df()

,category,restaurants,avg_stars
0,Coffee Roasteries,140,4.31
1,Poke,187,4.29
2,Food Trucks,1008,4.22
3,Vegan,856,4.22
4,Farmers Market,122,4.20
...,...,...,...
125,Chicken Shop,808,3.07
126,Chicken Wings,2966,2.97
127,Burgers,5636,2.93
128,Tex-Mex,919,2.92


Among the 130 categories with at least 100 restaurants, the highest average ratings
are Coffee Roasteries (4.31 stars, 140 restaurants), Poke (4.29, 187), Food Trucks (4.22, 1,008),
Vegan (4.22, 856) and Farmers Market (4.20, 122). The lowest are Fast Food (2.63, 6,472
restaurants), Tex-Mex (2.92, 919), Burgers (2.93, 5,636), Chicken Wings (2.97, 2,966) and
Chicken Shop (3.07, 808). The gap between the top and bottom is 1.68 stars.

The top is made of specialist and independent food types (coffee, poke, vegan, food trucks), and
the bottom of casual, chain-heavy ones (fast food, burgers, wings). Fast Food and Burgers are
also among the biggest categories in the dataset, so the low ratings are for thousands of
restaurants, not a few.

## Q4. what is inside attributes, and the link to stars

## Q5. price range and stars, and the most expensive cuisines

## Q6. chains vs independents

## Q7. restaurants with the most reviews

# Open and closed

## Q8. stars and share closed, by star band

## Q9. how long restaurants last, open vs closed

# Time and reviewers

## Q10. reviews per year

## Q11. average stars per year

## Q12. heavy vs occasional reviewers

# New Orleans review text

## Q13. review length vs stars

## Q14. common words in low vs high star reviews